# Langchain Conversational Memory and Fact Extraction Bot
Welcome to this tutorial on building an advanced conversational chatbot using Langchain! 

This bot uses **Ollama** for local inference (Mistral for text generation, nomic-embed-text for embeddings) and **Chroma DB** to manage two kinds of memories:
1. **Facts**: Durable semantic facts extracted from conversation.
2. **Summaries**: Rolling summaries of recent conversation chunks.

First, let's start by importing all the necessary modules.

In [55]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder
from langchain_chroma import Chroma
from langchain_ollama import  OllamaEmbeddings
from langchain_core.messages import HumanMessage,AIMessage
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

### 1. Initialize the Language Model
We initialize our Chat Model using Langchain's `init_chat_model`. Here, we use the `mistral` model hosted locally via `ollama`.

In [2]:
model = init_chat_model(model="mistral",model_provider="ollama")

### 2. Define the Main Chat Prompt
The main prompt tells the assistant how to behave. Crucially, we pass in dynamically retrieved `{facts}` and `{memories}` (summaries) alongside recent messages.

In [3]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a helpful AI assistant.

Use the provided memories when relevant.
Prefer recent conversation over old memories if they conflict.

Relevant Facts:
{facts}

Relevant Memories:
{memories}
"""
    ),

    MessagesPlaceholder("recent_messages"),

    (
        "human",
        "{user_message}"
    ),
])


### 3. Summary and Fact Prompts
To prevent the context window from exploding, we define prompts to summarize older messages and extract durable facts. 

Below is the **Summary Prompt**:

In [4]:
summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a conversation summarizer.

Create a concise summary of important events,
decisions, discussions, and outcomes.

Ignore greetings and small talk.
Focus on information useful for future conversations.
"""
    ),
    (
        "human",
        "{conversation_chunk}"
    )
])

Below is the **Fact Extraction Prompt**. Notice how we instruct the model to return a structured JSON representing the extracted facts.

In [41]:
fact_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a semantic memory extraction system.

Extract durable facts that may be useful in future conversations.

Rules:
- Keep only stable information.
- Ignore temporary information.
- Ignore greetings.
- Ignore one-time questions.
- Ignore speculation.
- Return JSON.

Output schema:

[
  {{
    "type": "...",
    "fact": "...",
    "confidence": 0.0
  }}
]
"""
    ),
    (
        "human",
        "{conversation_chunk}"
    )
])

In [42]:
print(fact_prompt.input_variables)

['conversation_chunk']


### 4. Memory Retrieval Prompt
When a user asks a question, we need to know *what* to search for in our database. This prompt helps generate a targeted search query based on recent conversation.

In [58]:
retrieval_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a memory retrieval assistant.

Generate a search query that can retrieve
relevant memories for answering the user's latest message.
only generate query no introduction or endings only query
"""
    ),

    MessagesPlaceholder("recent_messages"),

    (
        "human",
        "{user_message}"
    )
])

### 5. Assembling Chains
We can use LCEL (LangChain Expression Language) to chain together inputs, prompts, and models.

First, the **Main Chain** to generate the final response:

In [44]:
main_chain = (
    {
        "facts":lambda x : x["facts"],
        "memories":lambda x:x["summary"],
        "recent_messages":lambda x:x["recent_messages"],
        "user_message":lambda x:x["user_input"] 
    }|
    prompt | model )

Next, the **Summary Model Chain** and **Fact Model Chain** that will process chunks of conversation to build long-term memory:

In [45]:
summary_model =  (
     {
        "conversation_chunk":lambda x : x["recent_messages"]
    }|
    summary_prompt | 
    model)

fact_model = (
    {
        "conversation_chunk":lambda x : x["recent_messages"]
    }|
    fact_prompt |
    model)

### 6. Embeddings and Vector Stores
We need a way to store our facts and summaries so they can be retrieved semantically. We'll use local `OllamaEmbeddings` and Chroma.

In [46]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

Initialize the **Summary Vector Store**:

In [10]:
summary_store = Chroma(
    collection_name = "summary_collection",
    embedding_function = embeddings,
    persist_directory="./chroma_db"
)

Initialize the **Fact Vector Store**:

In [11]:
fact_store = Chroma(
    collection_name = "fact_collection",
    embedding_function = embeddings,
    persist_directory="./chroma_db"
)

### 7. Helper Functions for Retrieval and Session State
To tie everything together, we create retriever functions that filter memories by `user_id` so data stays isolated.

In [12]:
def get_fact_retriever(user_id: str):
    return fact_store.as_retriever(
        search_kwargs={
            "k": 5,
            "filter": {"user_id": user_id}
        }
    )

In [13]:
def get_summary_store_retriever(user_id: str):
    return summary_store.as_retriever(
        search_kwargs={
            "k": 5,
            "filter": {"user_id": user_id}
        }
    )

We'll use an in-memory dictionary `store` to track active sessions, keeping recent `messages` and tracking metadata.

In [94]:
store = {}

In [ ]:
def get_session(session_id):
    if session_id not in store:
        store[session_id] = {"messages":[],"message_count":1,"summary":[],"facts":[],"meta_data":{"user_id":session_id}}
    return store[session_id]

### 8. The Core Chat Function
This is the heart of the system. The `chat` function does the following:
1. Loads recent messages.
2. Every 10 messages, triggers fact extraction and summarization, storing the results in Chroma.
3. Retrieves relevant historical facts and summaries using the user's latest input.
4. Invokes the `main_chain` to get the final response.

In [75]:
def chat(session_id,userInput):
    print("messages loaded!:")

    history = get_session(session_id)
    history["messages"].append(HumanMessage(userInput))
    message_count = history["message_count"] 
    recent_messages = history["messages"][-4:]

    print("summary reterievels loaded!:")

    summary_reteriever = get_summary_store_retriever(session_id)
    fact_reteriever = get_fact_retriever(session_id)

    if message_count % 10 == 0:
        print("fact models invoking:")

        fact = fact_model.invoke({"recent_messages":history["messages"][-10:]}).content
        summary = summary_model.invoke({"recent_messages":history["messages"][-10:]}).content

        fact_store.add_documents([
            Document(
                page_content=fact,
                metadata=history["meta_data"]
            )
        ])
        summary_store.add_documents([
            Document(
                page_content=summary,
                metadata=history["meta_data"]
            )
        ])

        history["summary"].append(summary)
        history["facts"].append(fact)
    
    print("messages are stored in stores creating chain:")
    load_fact_chain = (
        {
         "recent_messages": lambda x : x["recent_messages"],
         "user_message":lambda x:x["user_input"]   
        }|
        retrieval_prompt| model |
        StrOutputParser()|
        fact_reteriever)
    
    load_summary_chain = (
        {
         "recent_messages": lambda x : x["recent_messages"],
         "user_message":lambda x:x["user_input"]   
        }|
        retrieval_prompt |model|
        StrOutputParser()|
        summary_reteriever
        )
    print("chain_created invoking:")
    load_fact = load_fact_chain.invoke(
        {
            "recent_messages":history["messages"][-10:],
            "user_input":userInput
        }
    )

    load_summary = load_summary_chain.invoke(
        {
            "recent_messages":history["messages"][-10:],
            "user_input":userInput
        }
    )
    print("invoke completed main chain running:")
    response = main_chain.invoke(
        {
            "facts":load_fact,
            "summary":load_summary,
            "recent_messages":history["messages"][-4:],
            "user_input":userInput
        }
    )
    history["message_count"] += 2
    history["messages"].append(response)
    return response





### 9. Let's Test It Out!
We set up a mock `session_id` and run the chat function to see how it answers questions while leveraging memory.

In [70]:
session_id = "user1234"

In [84]:
chat_test = chat(session_id=session_id,userInput="what is AI")
chat_test

messages loaded!:
summary reterievels loaded!:
fact models invoking:
messages are stored in stores creating chain:
chain_created invoking:
invoke completed main chain running:


AIMessage(content=" Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. It's a broad field encompassing various subfields, such as machine learning, natural language processing, robotics, and computer vision. These technologies allow computers to perform tasks typically requiring human-like cognitive functions, such as problem-solving, decision making, and perception.\n\n(This response is the same as my previous answer because it contains the most up-to-date information provided.)", additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2026-06-13T05:29:16.2718135Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9567933100, 'load_duration': 76143600, 'prompt_eval_count': 665, 'prompt_eval_duration': 632513000, 'eval_count': 109, 'eval_duration': 8781115000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'}, id='lc_run--019ebf74-8b8f-74a3-9934-cbf250

In [95]:
store

{}

In [89]:
fact_store.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [92]:
summary_store.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [ ]:
summary_store.delete(ids=summary_store.get()["ids"]) # to delete the memory in the store
fact_store.delete(ids=fact_store.get()["ids"])x